# AI-Driven Student Performance Prediction System
## Academic Experimentation & Model Evaluation Notebook
**Course / Degree:** B.Tech Computer Science & Engineering  
**Objective:** Build and evaluate machine learning models for continuous grade estimation (Regression) and performance tier categorization (Classification).

### 1. Import Necessary Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
)

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
print("All required machine learning libraries imported successfully!")

### 2. Load the Dataset

In [ ]:
data_path = os.path.join("..", "data", "student_performance.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
display(df.head())

### 3. Exploratory Data Analysis (EDA) & Summary Statistics

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics ---")
display(df.describe().round(2))

print("\n--- Performance Category Breakdown ---")
print(df["performance_category"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Marks distribution
sns.histplot(df["final_marks"], kde=True, color="#3498db", ax=axes[0])
axes[0].set_title("Distribution of Final Marks (Target: Regression)")
axes[0].axvline(df["final_marks"].mean(), color="red", linestyle="--", label="Mean")
axes[0].legend()

# Category distribution
sns.countplot(data=df, x="performance_category", hue="performance_category", order=["Low", "Average", "High"], palette="Set2", legend=False, ax=axes[1])
axes[1].set_title("Count of Students per Performance Category (Target: Classification)")

plt.tight_layout()
plt.show()

### 4. Correlation Analysis

In [ ]:
plt.figure(figsize=(9, 7))
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Academic Features")
plt.show()

### 5. Data Preprocessing & Leak-Free Train-Test Splitting

In [ ]:
# Encode categorical variable
df_encoded = df.copy()
df_encoded["extracurricular_activity"] = df_encoded["extracurricular_activity"].map({"Yes": 1, "No": 0})

# Feature columns
features = [
    "attendance", "previous_marks", "study_hours", "assignment_score",
    "internal_marks", "completed_assignments", "class_participation", "extracurricular_activity"
]

X = df_encoded[features]
y_reg = df_encoded["final_marks"]
y_clf = df_encoded["performance_category"]

# 80/20 Train-Test Split with fixed random seed
X_train, X_test, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X, y_reg, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

# Standard Scaling: Fit ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### 6. Regression Modeling (Predicting Final Marks)
We compare **Linear Regression** and **Random Forest Regressor** using MAE, MSE, RMSE, and $R^2$ Score.

In [ ]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
}

reg_metrics = []
for name, model in reg_models.items():
    model.fit(X_train_scaled, y_train_reg)
    preds = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test_reg, preds)
    mse = mean_squared_error(y_test_reg, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_reg, preds)
    
    reg_metrics.append({
        "Model": name,
        "MAE": round(mae, 4),
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "R2_Score": round(r2, 4)
    })

reg_summary_df = pd.DataFrame(reg_metrics).set_index("Model")
print("--- Regression Comparison ---")
display(reg_summary_df)

### 7. Classification Modeling (Categorizing Performance: Low, Average, High)
We compare **Logistic Regression**, **Decision Tree**, and **Random Forest Classifier** using Accuracy, Precision, Recall, and F1-Score.

In [ ]:
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

clf_metrics = []
for name, model in clf_models.items():
    model.fit(X_train_scaled, y_train_clf)
    preds = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test_clf, preds)
    prec = precision_score(y_test_clf, preds, average="weighted", zero_division=0)
    rec = recall_score(y_test_clf, preds, average="weighted", zero_division=0)
    f1 = f1_score(y_test_clf, preds, average="weighted", zero_division=0)
    
    clf_metrics.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1_Score": round(f1, 4)
    })

clf_summary_df = pd.DataFrame(clf_metrics).set_index("Model")
print("--- Classification Comparison ---")
display(clf_summary_df)

### 8. Confusion Matrix for Best Classifier

In [ ]:
best_clf = LogisticRegression(max_iter=1000, random_state=42)
best_clf.fit(X_train_scaled, y_train_clf)
y_preds = best_clf.predict(X_test_scaled)

labels = ["Low", "Average", "High"]
cm = confusion_matrix(y_test_clf, y_preds, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - Best Classifier (Logistic Regression)")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.show()

print("\nClassification Report:")
print(classification_report(y_test_clf, y_preds, labels=labels, zero_division=0))